# Plexus — Step-1 Validation Notebook

**Purpose (per context §8): prove the core graph is REAL and LEGIBLE on the 2025 substrate BEFORE choosing a stack or scaffolding any app.** No framework decision here. This notebook only answers: *does a sensible, stratified, skill-overlap role graph emerge from this messier 2025 file?*

**Three PASS/FAIL gates decide the project:**
1. **Clustering** — are role-to-role nearest neighbours sensible (Testing↔QA, Developer↔Lead↔Architect, DBA↔Database Architect)? Mush ⇒ FAIL.
2. **Stratification** — do services-vs-GCC produce visibly distinguishable nodes for high-volume roles? If GCC is too sparse to split most roles, that is the honest §7 finding — record it, don't paper over it.
3. **Bridge** — does a within-stratum bridge-skill computation return a non-trivial, defensible reachable-set expansion?

**Deliberate Step-1 method choices (flagged for review):**
- **Skill normalisation is lean rule-based** (lowercase/strip + curated alias map), NOT embedding-based. Step-1's job is to prove structure holds, not ship production NLP. Embedding resolution is the documented upgrade.
- **Role nodes are rule-based canonical roles**, not KMeans clusters. This maximises legibility and isolates the *structure* question from clustering-hyperparameter noise. An optional unsupervised clustering cross-check is appended (honours §8's "dual-field clustering" intent). If both agree → strong PASS.
- **`Software Engineer (General)` is de-hubbed** (Step 3b): generically-titled postings are re-routed to a specific role when their skills clearly signal one, so the generic bucket stops acting as a hub adjacent to everything.
- **Employer-type tagging** runs the §6 services blocklist FIRST with **token-boundary matching** (substring matching falsely deletes Morgan Stanley / Honeywell / Ashok Leyland via the "ey" substring), then genuine-GCC match, then a staffing-agency heuristic, then `unknown`.
- **Gate 3 uses a skill-COVERAGE bridge, not a cosine-vector boost.** A single skill cannot move a dense aggregate cosine enough to bridge an isolated role; coverage (how much of a target role's core skills you'd hold) is both faithful to §4 and produces nameable, multi-skill upskilling paths. It also exposes the honest truth that specialist roles are skill-islands.

**Verified inputs (byte-level, this session):**
- `indian-job-market-dataset-2025.xlsx` — Sheet1, 97,929 × 17. Broad cross-function. `jobUploaded` = relative strings (no temporal use). Salary "Not disclosed" is a string (65.4%); `minimumSalary == 0` means undisclosed (we use NO salary deliverables).
- `GCC-Journal-India-List.xlsx` — sheet `All GCCs – India Master`, 274 × 9. 27 are services conflators (drop), 247 genuine captives.

**Expected first-run numbers (validated end-to-end this session):** IT subset 27,700 (28.3%); de-hub re-routes ≈ 1,196 postings; employer mix ≈ services 30%, GCC 4.4%, agency 4.5%, unknown 61% (agency is lower than a first cut because the heuristic is deliberately staffing-specific to avoid mislabelling services firms); Gate 3 ≈ 11 connected / 2 reachable-via-upskill / 7 island.


## 0 · Config & imports

Edit `SUBSTRATE` / `GCC_FILE` if your paths differ. Only standard data-science deps (pandas, numpy, scikit-learn, scipy, networkx). No network / model downloads.

In [15]:
import re
import numpy as np
import pandas as pd
from collections import Counter, defaultdict
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import networkx as nx

pd.set_option("display.max_colwidth", 60)

# --- paths (edit if needed) ---
SUBSTRATE = "indian-job-market-dataset-2025.xlsx"
GCC_FILE  = "GCC-Journal-India-List.xlsx"

# --- tunables ---
MIN_ROLE_POSTINGS = 80     # a canonical role needs >= this many postings to be a graph node
MIN_STRATUM       = 25     # a (role, employer-type) stratum needs >= this many postings to be a stratified node
EDGE_THRESHOLDS   = [0.20, 0.30, 0.40, 0.50]  # cosine thresholds to sweep for the graph
GRAPH_THRESHOLD   = 0.30   # working threshold used after the sensitivity sweep

# --- bridge / coverage tunables (Gate 3) ---
BRIDGE_TOPK   = 12         # how many core skills define a role's "worker" profile
COV_THR       = 0.30       # a role is reachable if you cover >= this fraction of its core skills
BRIDGE_STEPS  = 3          # max skills in a single upskilling path
ISLAND_FLOOR  = 0.15       # below this best-coverage, a role is a true specialist silo

# generic non-skills that otherwise act as fake bridges / pollute role cores
SKILL_STOP = {'software','development','software development','software engineering','programming','coding',
 'technical','troubleshooting','communication','good english communication','english','analytical',
 'problem solving','teamwork','sales','administration','management','it','bi','design','testing',
 'engineering','operations','support','documentation','good communication','interpersonal skills',
 'time management','leadership','agile','scrum','team management','proof of concept','client interaction',
 'information technology','computer science'}


## 1 · Load & confirm shapes

Reads the actual files — does not trust labels.

In [16]:
df  = pd.read_excel(SUBSTRATE, sheet_name="Sheet1")
gcc = pd.read_excel(GCC_FILE, sheet_name="All GCCs \u2013 India Master")  # en-dash in sheet name

print("Substrate:", df.shape, "| cols:", list(df.columns))
print("GCC master:", gcc.shape, "| cols:", list(gcc.columns))
assert {"title","tagsAndSkills","companyName"} <= set(df.columns), "substrate columns changed"
assert "Company / GCC Name" in gcc.columns, "GCC name column changed"


Substrate: (97929, 17) | cols: ['title', 'jobId', 'currency', 'jobUploaded', 'companyName', 'tagsAndSkills', 'experience', 'salary', 'location', 'companyId', 'ReviewsCount', 'AggregateRating', 'jobDescription', 'minimumSalary', 'maximumSalary', 'minimumExperience', 'maximumExperience']
GCC master: (274, 9) | cols: ['No.', 'Company / GCC Name', 'Website', 'Parent HQ Country', 'City', 'Sector', 'India Headcount', 'Key Roles Hired', 'GCC Notes']


## 2 · Step 1 — Filter the IT/tech subset

Dual-field, deliberately **over-inclusive** (false rejects are silent & costly; false accepts self-correct downstream). A posting is IT if its **title** matches a tech-role keyword OR its **tagsAndSkills** carries ≥ 2 tech-skill tokens.

In [17]:
TECH_TITLE = ['software','developer','programmer','full stack','fullstack','frontend','front end',
'backend','back end','devops',' sre','data scien','data engineer','data analyst','ml engineer',
'machine learning',' ai ','ai engineer','qa ','sdet','test engineer','automation test','tester',
'web develop','mobile develop','android','ios develop','java develop','python develop','dot net',
'.net','php develop','cloud engineer','cloud architect','solution architect','technical architect',
'database admin','dba','system admin','network engineer','security engineer','cyber','sap ',
'salesforce','tech lead','engineering manager','data architect','etl','bi develop','platform engineer']

TECH_SKILL = ['java','python','javascript','typescript','react','angular','vue','node','express',
'aws','azure','gcp','kubernetes','docker','sql','mysql','postgres','mongodb','spring','hibernate',
'c++','c#','.net','php','html','css','git','jenkins','kafka','spark','hadoop','tensorflow','pytorch',
'selenium','rest api','microservices','devops','linux','terraform','redis','elasticsearch','scala','golang','rust']

t = df["title"].fillna("").str.lower()
s = df["tagsAndSkills"].fillna("").str.lower()
title_hit  = t.apply(lambda x: any(k in x for k in TECH_TITLE))
skill_hits = s.apply(lambda x: sum(1 for k in TECH_SKILL if k in x))

is_it = title_hit | (skill_hits >= 2)
sub = df[is_it].copy().reset_index(drop=True)

print(f"Corpus:    {len(df)}")
print(f"IT subset: {len(sub)} ({len(sub)/len(df)*100:.1f}%)")
print(f"  via title match: {title_hit.sum()} | via >=2 tech skills: {(skill_hits>=2).sum()}")
print("\nKEPT sample:");    [print("  +", str(x)[:70]) for x in sub['title'].head(8)]
print("\nDROPPED sample:"); [print("  -", str(x)[:70]) for x in df[~is_it]['title'].head(8)]


Corpus:    97929
IT subset: 27700 (28.3%)
  via title match: 22957 | via >=2 tech skills: 16522

KEPT sample:
  + Site Reliability Engineer
  + 4LPA CTC For Email Chat Backend & Non Voice Process For Kolkata
  + ATM Implementation - Software Engineer / Senior Software Engineer
  + Software Engineer Implementation
  + Embedded Software Engineering Specialist
  + Senior Software Engineer
  + Sr.Lead Software Engineer
  + PHP Developer

DROPPED sample:
  - Sr. HR Recruiter (NON IT)
  - Fire And Safety Officer
  - Opening For Performance Marketing - Chennai
  - Medical Billing Executive
  - Senior Group Product Manager -  CNS Therapy
  - Senior Accountant
  - Maintenance Technician
  - Officer/Engineer Technical


[None, None, None, None, None, None, None, None]

## 3 · Step 2 — Skill normalisation (lean rule-based)

Split `tagsAndSkills` on commas, lowercase/strip, then collapse high-frequency variants via an alias map (react/reactjs/react.js → react, etc.). Reports top skills before vs after so you can see the collapse working. Add aliases as the top list reveals them.

In [18]:
ALIAS = {
    'reactjs':'react','react.js':'react','react js':'react','react native':'react native',
    'nodejs':'node','node.js':'node','node js':'node',
    'js':'javascript','ts':'typescript',
    'k8s':'kubernetes','aws cloud':'aws','amazon web services':'aws',
    'ms sql':'sql server','sql server':'sql server','mssql':'sql server',
    'postgresql':'postgres','mongo':'mongodb',
    'dot net':'.net','dotnet':'.net','asp.net':'.net','c sharp':'c#','csharp':'c#',
    'spring boot':'spring boot','springboot':'spring boot','spring':'spring',
    'rest':'rest api','restful':'rest api','rest apis':'rest api','restful api':'rest api',
    'ci/cd':'ci cd','cicd':'ci cd',
    'gen ai':'generative ai','genai':'generative ai','llm':'llms','llm models':'llms',
    'ml':'machine learning','a.i.':'ai','artificial intelligence':'ai',
    'power bi':'power bi','powerbi':'power bi',
    'html5':'html','css3':'css',
}

def clean_tok(tok):
    tok = tok.strip().lower()
    tok = re.sub(r'[^a-z0-9+#. /]', ' ', tok)
    tok = re.sub(r'\s+', ' ', tok).strip()
    return ALIAS.get(tok, tok)

def parse_skills(cell):
    if not isinstance(cell, str): return []
    out = []
    for raw in cell.split(','):
        c = clean_tok(raw)
        if c and len(c) > 1:
            out.append(c)
    return out

sub["skills"] = sub["tagsAndSkills"].apply(parse_skills)

raw_counter  = Counter(x.strip().lower() for cell in sub['tagsAndSkills'].dropna() for x in str(cell).split(','))
norm_counter = Counter(sk for lst in sub['skills'] for sk in lst)

print("avg skills/posting:", round(sub['skills'].apply(len).mean(), 1))
print("\nTOP 25 NORMALISED skills:")
for sk, c in norm_counter.most_common(25):
    print(f"  {sk:28s} {c}")
print("\nreact-family collapse check:", {k:v for k,v in raw_counter.items() if 'react' in k})


avg skills/posting: 7.8

TOP 25 NORMALISED skills:
  python                       5322
  css                          4080
  java                         3977
  sql                          3380
  sap                          3028
  javascript                   3022
  c#                           2586
  development                  2096
  rest api                     2039
  aws                          1935
  c++                          1696
  html                         1577
  continuous integration       1566
  react                        1556
  spring boot                  1540
  microservices                1535
  kubernetes                   1471
  agile                        1402
  application development      1393
  software development         1384
  software testing             1302
  web services                 1285
  project management           1201
  oracle                       1169
  automation                   1166

react-family collapse check: {'react.js': 1281, 

## 4 · Step 3 — Role extraction (canonical roles) + explicit unclassified node

Each posting is assigned to the **first** canonical role whose keyword matches its title (specific roles ordered before general). No match ⇒ `unclassified` — the generalist node, **excluded from the graph** (reported, never silently dropped). Canonical roles with ≥ `MIN_ROLE_POSTINGS` become graph nodes.

Tune `CANONICAL_ROLES` freely — the ordering matters (Full Stack before Front/Back end; Data Scientist before generic).

In [19]:
# (canonical label, [keywords searched in title]) — ORDER = priority, specific first
CANONICAL_ROLES = [
    ('Data Scientist / ML',        ['data scientist','machine learning','ml engineer','ai engineer','deep learning','data science']),
    ('Data Engineer',              ['data engineer','etl','big data','hadoop','spark develop','databricks']),
    ('Data / BI Analyst',          ['data analyst','business analyst','bi develop','bi analyst','power bi','tableau','analytics']),
    ('DevOps / SRE',               ['devops','site reliability',' sre','platform engineer','build and release','cloudops']),
    ('Cloud Engineer / Architect', ['cloud engineer','cloud architect','aws engineer','azure engineer','cloud consultant']),
    ('Security Engineer',          ['security engineer','cyber','infosec','soc analyst','penetration','vapt']),
    ('QA / Test Engineer',         ['qa','quality assurance','test engineer','sdet','automation test','software tester','testing','test analyst']),
    ('Full Stack Developer',       ['full stack','fullstack','mean stack','mern','mern stack']),
    ('Frontend Developer',         ['frontend','front end','react develop','angular develop','ui developer','ui engineer','ui/ux develop']),
    ('Mobile Developer',           ['android develop','ios develop','mobile develop','flutter','react native develop']),
    ('.NET Developer',             ['.net','dot net','c# develop','asp.net']),
    ('PHP / Web Developer',        ['php develop','laravel','wordpress developer','codeigniter','core php']),
    ('Backend / Java Developer',   ['backend','back end','java develop','spring develop','node develop','python develop','golang develop','api develop']),
    ('Database Architect',         ['database architect','data architect']),
    ('Database Administrator',     ['dba','database admin','oracle dba','sql dba','database engineer']),
    ('Solution / Tech Architect',  ['solution architect','technical architect','software architect','enterprise architect']),
    ('Eng Manager / Tech Lead',    ['engineering manager','tech lead','technical lead','team lead','development lead','delivery lead']),
    ('SAP Consultant',             ['sap']),
    ('Salesforce Developer',       ['salesforce']),
    ('Network / Sys Admin',        ['network engineer','system administrator','system admin','network admin','windows admin','linux admin','noc']),
    ('Software Engineer (General)',['software engineer','software developer','programmer','sde','application developer','it developer']),
]

def assign_role(title):
    tl = str(title).lower()
    for label, kws in CANONICAL_ROLES:
        if any(k in tl for k in kws):
            return label
    return 'unclassified'

sub["role"] = sub["title"].apply(assign_role)

role_counts = sub["role"].value_counts()
uncl = role_counts.get("unclassified", 0)
print(f"unclassified (excluded from graph): {uncl} ({uncl/len(sub)*100:.1f}%)")
print(f"\nrole distribution ({len(role_counts)-1} candidate roles):")
print(role_counts.to_string())


unclassified (excluded from graph): 8428 (30.4%)

role distribution (21 candidate roles):
role
unclassified                   8428
Software Engineer (General)    3874
SAP Consultant                 3105
Full Stack Developer           1721
Backend / Java Developer       1388
Data Engineer                  1337
QA / Test Engineer             1267
Data Scientist / ML             941
DevOps / SRE                    898
Eng Manager / Tech Lead         705
.NET Developer                  704
Solution / Tech Architect       572
Data / BI Analyst               567
Frontend Developer              426
Security Engineer               377
Salesforce Developer            289
Mobile Developer                262
Network / Sys Admin             257
Database Administrator          219
PHP / Web Developer             132
Cloud Engineer / Architect      131
Database Architect              100


### Step 3b — De-hub the `Software Engineer (General)` blob

`Software Engineer (General)` catches every generically-titled posting and becomes a hub that sits ~0.3–0.4 from everything (confirmed: it was the largest KMeans cluster and the nearest neighbour of most roles). We re-route a posting OUT of the generic bucket when its **skills** clearly signal a specific role (≥ 2 signature-skill hits AND a clear winner over the runner-up). Transparent and tunable — edit `SIGNALS`.

In [20]:
SIGNALS = {
 'Frontend Developer':       ['react','angular','vue','javascript','typescript','redux','rxjs','html','css','responsive web design','frontend development','next js'],
 'Backend / Java Developer': ['spring','spring boot','hibernate','microservices','j2ee','jpa','rest api','spring mvc','kafka'],
 '.NET Developer':           ['.net','net core','c#','asp','mvc','entity framework','wpf'],
 'PHP / Web Developer':      ['php','laravel','wordpress','codeigniter','magento'],
 'Data Scientist / ML':      ['machine learning','deep learning','tensorflow','pytorch','keras','nlp','computer vision','data scientist','pandas'],
 'Data Engineer':            ['etl','spark','hadoop','pyspark','hive','airflow','snowflake','data warehousing','scala','databricks'],
 'Data / BI Analyst':        ['power bi','tableau','data analysis','data visualization','qlik'],
 'DevOps / SRE':             ['devops','kubernetes','docker','terraform','jenkins','ci cd','ansible','sre','aws devops'],
 'QA / Test Engineer':       ['selenium','automation testing','sdet','appium','cypress','playwright','rest assured','test automation','manual testing'],
 'Database Administrator':   ['oracle dba','rman','oracle rac','database administration','sql server dba'],
 'SAP Consultant':           ['sap','sap abap','sap sd','sap fico','sap hana','sap mm'],
 'Salesforce Developer':     ['salesforce','apex','visualforce','soql','salesforce lightning'],
 'Mobile Developer':         ['android','ios','flutter','kotlin','swift','react native'],
 'Security Engineer':        ['penetration testing','vapt','siem','soc','cyber security','owasp'],
}

def reroute(skills):
    sk = set(skills); best=None; bn=0; second=0
    for role, sig in SIGNALS.items():
        h = sum(1 for x in sig if x in sk)
        if h > bn: second=bn; bn=h; best=role
        elif h > second: second=h
    return best if (bn >= 2 and bn - second >= 1) else None

mask = sub["role"] == "Software Engineer (General)"
rr = sub.loc[mask, "skills"].apply(reroute)
n_before = int(mask.sum())
sub.loc[mask, "role"] = rr.where(rr.notna(), "Software Engineer (General)")
n_after = int((sub["role"]=="Software Engineer (General)").sum())
print(f"de-hub: Software Engineer (General) {n_before} -> {n_after}  (rerouted {n_before-n_after})")
print("rerouted into:", rr.dropna().value_counts().to_dict())

# (re)compute role nodes AFTER de-hub
role_counts = sub["role"].value_counts()
node_roles = [r for r,c in role_counts.items() if r != 'unclassified' and c >= MIN_ROLE_POSTINGS]
print(f"\ngraph-eligible role nodes (>= {MIN_ROLE_POSTINGS} postings): {len(node_roles)}")


de-hub: Software Engineer (General) 3874 -> 2678  (rerouted 1196)
rerouted into: {'Frontend Developer': 400, 'SAP Consultant': 238, 'Backend / Java Developer': 203, 'Data Engineer': 91, 'DevOps / SRE': 84, '.NET Developer': 72, 'Salesforce Developer': 47, 'QA / Test Engineer': 21, 'Mobile Developer': 18, 'Data / BI Analyst': 12, 'PHP / Web Developer': 4, 'Data Scientist / ML': 4, 'Database Administrator': 1, 'Security Engineer': 1}

graph-eligible role nodes (>= 80 postings): 21


### Build role → skill profiles (TF-IDF)

TF-IDF over each role's pooled skill tokens. TF-IDF **down-weights ubiquitous hub skills** (sql, communication) automatically — the §6 hub-normalisation concern, handled at the weighting layer. These profile vectors drive the NN check, the graph edges, and the bridge.

In [21]:
role_docs = {}
for r in node_roles:
    toks = [sk for lst in sub.loc[sub.role==r, "skills"] for sk in lst]
    role_docs[r] = " ".join(t.replace(" ", "_") for t in toks)   # keep multiword skills intact

roles = list(role_docs.keys())
vec = TfidfVectorizer(min_df=2, sublinear_tf=True)
X = vec.fit_transform([role_docs[r] for r in roles])
SIM = cosine_similarity(X)
print("role-profile matrix:", X.shape, "(roles x skill-vocab)")

def top_skills(role, k=8):
    i = roles.index(role)
    row = X[i].toarray().ravel()
    vocab = np.array(vec.get_feature_names_out())
    idx = row.argsort()[::-1][:k]
    return [vocab[j].replace("_"," ") for j in idx if row[j] > 0]


role-profile matrix: (21, 3561) (roles x skill-vocab)


## ✅ GATE 1 — Clustering / nearest-neighbour sanity

For each probe role, print its top-5 nearest neighbours by skill-profile cosine. **PASS** if neighbours are sensible (QA near Test/Automation; Backend near Full Stack/Architect; DBA near Database Architect/Data Engineer). **FAIL** = mush (unrelated roles adjacent) ⇒ rethink before any app.

*This is a human-judgment gate — read the output.*

In [22]:
def neighbours(role, k=5):
    if role not in roles: return f"(role '{role}' not a node)"
    i = roles.index(role)
    order = SIM[i].argsort()[::-1]
    out = []
    for j in order:
        if roles[j] != role:
            out.append((roles[j], round(float(SIM[i][j]), 2)))
        if len(out) == k: break
    return out

PROBES = ['QA / Test Engineer','Backend / Java Developer','Database Administrator',
          'Data Scientist / ML','DevOps / SRE','Frontend Developer']
for p in PROBES:
    print(f"\n{p}")
    print("   top skills:", top_skills(p))
    print("   nearest   :", neighbours(p))



QA / Test Engineer
   top skills: ['test engineering', 'test case design', 'appium', 'playwright', 'api testing', 'quality assurance', 'qc', 'test scripts']
   nearest   : [('Software Engineer (General)', 0.4), ('Backend / Java Developer', 0.33), ('Eng Manager / Tech Lead', 0.32), ('DevOps / SRE', 0.3), ('Full Stack Developer', 0.29)]

Backend / Java Developer
   top skills: ['spring mvc', 'good english communication', 'boot', 'java development', 'spring boot', 'liferay', 'j2ee', 'microservices']
   nearest   : [('Full Stack Developer', 0.63), ('Frontend Developer', 0.44), ('Software Engineer (General)', 0.43), ('DevOps / SRE', 0.42), ('Eng Manager / Tech Lead', 0.41)]

Database Administrator
   top skills: ['rman', 'oracle dba', 'mysql database administration', 'oracle rac', 'database administration', 'rac', 'recovery', 'database security']
   nearest   : [('DevOps / SRE', 0.27), ('Data Engineer', 0.25), ('Software Engineer (General)', 0.22), ('Database Architect', 0.21), ('Backend /

## 5 · Step 4 — Employer-type tagging (blocklist FIRST, token-boundary)

Order: **services blocklist (token match)** → **genuine GCC** → **staffing agency** → **unknown**. Token-boundary matching is mandatory (substring "ey" otherwise eats Morgan Stanley / Honeywell / Ashok Leyland). Blocklist includes spelled-out variants (TCS posts as *Tata Consultancy Services*). Agency keywords are staffing-specific to avoid mislabeling services/consulting firms.

`unknown` holds genuine Indian product/startup + SMEs — **unseparable without a product-company reference list** (honest §7 finding, not a bug).

In [23]:
def norm(x):
    x = str(x).lower(); x = re.sub(r'[^a-z0-9 ]',' ',x); return re.sub(r'\s+',' ',x).strip()

def tok_match(name_norm, phrase):
    p = norm(phrase)
    return bool(p) and re.search(r'(?:^| )'+re.escape(p)+r'(?:$| )', name_norm) is not None

BLOCK = ['accenture','wipro','capgemini','cognizant','hcltech','hcl technologies','tcs',
'tata consultancy','tata consultancy services','tech mahindra','mphasis','hexaware','zensar',
'mindtree','ltimindtree','ltts','l t technology','persistent','coforge','birlasoft','cybage',
'conduent','genpact','wns','firstsource','ey','deloitte','kpmg','pwc','price waterhouse','ibm',
'dxc','atos','infosys','info edge','nagarro','happiest minds','sonata','cyient','kpit','zoho']

AGENCY = ['manpower','staffing','recruitment','placement','placements','hr solutions','hr services',
'recruiters','resourcing','workforce','job consultancy','manpower consultants','rpo','executive search']

def gcc_brand(raw):
    x = norm(raw)
    for junk in ['india','bangalore','bengaluru','hyderabad','pune','mumbai','chennai','delhi','ncr',
                 'gurgaon','gurugram','kolkata','gcc','global','capability','centre','center',
                 'technology','technologies','solutions','services','engineering','advanced','development']:
        x = re.sub(r'(?:^| )'+junk+r'(?:$| )', ' ', ' '+x+' ').strip()
    return re.sub(r'\s+',' ', x).strip()

# genuine GCC brand cores = GCC list minus services conflators
genuine = set()
for r in gcc["Company / GCC Name"].dropna():
    if not any(tok_match(norm(r), b) for b in BLOCK):
        b = gcc_brand(r)
        if len(b) >= 3:
            genuine.add(b)
genuine = sorted(genuine)
print(f"genuine GCC brand cores: {len(genuine)}")

def tag(cn):
    nm = norm(cn)
    if any(tok_match(nm, b) for b in BLOCK):   return 'services'
    if any(tok_match(nm, g) for g in genuine): return 'gcc'
    if any(a in nm for a in AGENCY):           return 'agency'
    return 'unknown'

sub["emp_type"] = sub["companyName"].apply(tag)
vc = sub["emp_type"].value_counts()
print(f"\nEmployer-type shares on IT subset (n={len(sub)}):")
for k in ['services','gcc','agency','unknown']:
    c = vc.get(k,0); print(f"  {k:9s}: {c:6d}  ({c/len(sub)*100:5.1f}%)")
print("\nGCC-tagged sample:", sub[sub.emp_type=='gcc']['companyName'].value_counts().head(8).index.tolist())
print("agency-tagged sample:", sub[sub.emp_type=='agency']['companyName'].value_counts().head(6).index.tolist())
print("\n[sanity] TCS rows ->", sub[sub.companyName.fillna('').str.lower().str.contains('tata consultancy')]['emp_type'].value_counts().to_dict())


genuine GCC brand cores: 230

Employer-type shares on IT subset (n=27700):
  services :   8236  ( 29.7%)
  gcc      :   1206  (  4.4%)
  agency   :   1241  (  4.5%)
  unknown  :  17017  ( 61.4%)

GCC-tagged sample: ['JPMorgan Chase Bank', 'ABB', 'Hsbc', 'Mastercard', 'Amazon', 'S&P Global Market Intelligence', 'Deutsche Bank', 'Oracle']
agency-tagged sample: ['Krazy Mantra HR Solutions Pvt. Ltd', 'Gratitude India Manpower Consultants Pvt. Ltd.', 'Venpa Staffing', 'White Horse Manpower Consultancy', 'Talent Corner Hr Services', 'SP Staffing']

[sanity] TCS rows -> {'services': 159}


## ✅ GATE 2 — Stratification (the moat)

Node-level stratification: `Role @ Services` and `Role @ GCC` are **different nodes**, built from employer-type-filtered postings. For each high-volume role with enough postings in BOTH strata (≥ `MIN_STRATUM`), compare the two skill profiles. **PASS** if they're visibly distinguishable for high-volume roles. If GCC is too sparse to split most roles, **that is the honest §7 finding** — print it plainly.

In [24]:
def stratum_doc(role, et):
    toks = [sk for lst in sub.loc[(sub.role==role)&(sub.emp_type==et), "skills"] for sk in lst]
    return toks

splittable = []
for r in node_roles:
    n_serv = ((sub.role==r)&(sub.emp_type=='services')).sum()
    n_gcc  = ((sub.role==r)&(sub.emp_type=='gcc')).sum()
    if n_serv >= MIN_STRATUM and n_gcc >= MIN_STRATUM:
        splittable.append((r, n_serv, n_gcc))

print(f"roles splittable into Services AND GCC strata (>= {MIN_STRATUM} each): {len(splittable)}\n")
if not splittable:
    print(">> GCC too sparse to stratify most roles at this threshold.")
    print(">> HONEST FINDING (§7): genuine GCC ~4% of IT postings on Naukri; the GCC stratum")
    print(">> is a small minority on this surface. Record it; do not paper over it.")
else:
    def topn(toks, k=10):
        c = Counter(toks); tot = sum(c.values()) or 1
        return [(w, round(n/tot*100,1)) for w,n in c.most_common(k)]
    for r, ns, ng in splittable[:6]:
        sv = dict(topn(stratum_doc(r,'services')))
        gc = dict(topn(stratum_doc(r,'gcc')))
        only_serv = [w for w in sv if w not in gc][:5]
        only_gcc  = [w for w in gc if w not in sv][:5]
        print(f"{r}  (services n={ns}, gcc n={ng})")
        print(f"   services-distinctive: {only_serv}")
        print(f"   gcc-distinctive     : {only_gcc}\n")


roles splittable into Services AND GCC strata (>= 25 each): 13

SAP Consultant  (services n=795, gcc n=47)
   services-distinctive: ['sap abap', 'sap abap development', 'css', 'sap sd', 'application design']
   gcc-distinctive     : ['troubleshooting', 'risk management', 'sap s hana', 'sap ewm', 'erp']

Software Engineer (General)  (services n=1413, gcc n=205)
   services-distinctive: ['css', 'development methodologies', 'c#', 'sap', 'web services']
   gcc-distinctive     : ['agile', 'system design', 'coding', 'machine learning', 'automation']

Full Stack Developer  (services n=320, gcc n=47)
   services-distinctive: ['full stack', 'python']
   gcc-distinctive     : ['kubernetes', 'agile']

Backend / Java Developer  (services n=426, gcc n=56)
   services-distinctive: ['development', 'rest api', 'boot', 'j2ee', 'java development']
   gcc-distinctive     : ['sql', 'python', 'agile', 'kubernetes', 'oracle']

Data Engineer  (services n=298, gcc n=45)
   services-distinctive: ['etl', 'data 

## 6 · Step 5 — Skill-overlap graph + threshold sensitivity

Nodes = canonical role nodes (TF-IDF profiles). Edge between two roles when cosine ≥ threshold. The edge threshold is a **design choice with no external validation signal** — so we sweep it and report node/edge/component counts. Pick a threshold that keeps the graph connected-but-legible (not a hairball, not shattered).

In [25]:
print("threshold sensitivity sweep:")
print(f"{'thr':>5} {'edges':>6} {'avg_deg':>8} {'components':>11} {'isolated':>9}")
for thr in EDGE_THRESHOLDS:
    G = nx.Graph(); G.add_nodes_from(roles)
    for i in range(len(roles)):
        for j in range(i+1, len(roles)):
            if SIM[i][j] >= thr:
                G.add_edge(roles[i], roles[j], weight=round(float(SIM[i][j]),3))
    degs = [d for _,d in G.degree()]
    iso = sum(1 for d in degs if d==0)
    print(f"{thr:>5} {G.number_of_edges():>6} {np.mean(degs):>8.2f} "
          f"{nx.number_connected_components(G):>11} {iso:>9}")

# build the working graph at GRAPH_THRESHOLD
G = nx.Graph(); G.add_nodes_from(roles)
for i in range(len(roles)):
    for j in range(i+1, len(roles)):
        if SIM[i][j] >= GRAPH_THRESHOLD:
            G.add_edge(roles[i], roles[j], weight=round(float(SIM[i][j]),3))
print(f"\nworking graph @ {GRAPH_THRESHOLD}: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")
print("connector skills (RAW cross-role frequency — NOT tf-idf, which suppresses hubs):")
# connector = a real skill appearing across many distinct role nodes (the bridges live here)
xrole = Counter()
for r in roles:
    seen = {sk for lst in sub.loc[sub.role==r, "skills"] for sk in lst} - SKILL_STOP
    for sk in seen:
        xrole[sk] += 1
for sk, c in xrole.most_common(15):
    print(f"   {sk:24s} in {c}/{len(roles)} role profiles")


threshold sensitivity sweep:
  thr  edges  avg_deg  components  isolated
  0.2    120    11.43           1         0
  0.3     51     4.86           4         2
  0.4     15     1.43          12        10
  0.5      3     0.29          18        17

working graph @ 0.3: 21 nodes, 51 edges
connector skills (RAW cross-role frequency — NOT tf-idf, which suppresses hubs):
   senior                   in 21/21 role profiles
   it services              in 21/21 role profiles
   python                   in 21/21 role profiles
   aws                      in 21/21 role profiles
   project management       in 21/21 role profiles
   automation               in 21/21 role profiles
   ci cd                    in 20/21 role profiles
   css                      in 20/21 role profiles
   application development  in 20/21 role profiles
   cloud                    in 20/21 role profiles
   github                   in 20/21 role profiles
   sql                      in 20/21 role profiles
   devops        

## ✅ GATE 3 — Within-stratum bridge-skill paths

**Model (revised — the cosine-boost model was too weak):** a single skill added to a role's dense aggregate profile barely moves a cosine, so it can never bridge an isolated role. Instead we use the **skill-coverage** model, which is also exactly the §4 framing — *"this opens adjacency to N roles; we measure skill overlap, not hiring transitions."*

Represent a worker as their role's **core skill set** (top `BRIDGE_TOPK` skills in the Services stratum, generic non-skills stripped). A target role is **reachable** if you already cover ≥ `COV_THR` of its core skills. A **bridge** is the shortest sequence (≤ `BRIDGE_STEPS`) of *real* skills that brings a new role into reach.

Each role is classified:
- **CONNECTED** — reaches ≥ 1 role at baseline (the development/data continent).
- **REACHABLE-VIA-UPSKILL** — base 0, but a short skill path opens a door.
- **ISLAND** — no path within `BRIDGE_STEPS`; if best coverage < `ISLAND_FLOOR` it's a true specialist silo (QA, Security, Network…). **This is the honest §7 finding — services-firm specialist JDs are skill-siloed — not a failure.**

**PASS** if connected-cluster bridges are non-trivial and interpretable AND islands are labelled honestly (not faked).


In [26]:
# worker = a role's core skill set within the SERVICES stratum (generic non-skills stripped)
core = {}
for r in node_roles:
    toks = [sk for lst in sub.loc[(sub.role==r)&(sub.emp_type=='services'), "skills"]
            for sk in lst if sk not in SKILL_STOP]
    if len(toks) >= 30:
        core[r] = set(s for s,_ in Counter(toks).most_common(BRIDGE_TOPK))
serv_roles = list(core)

def cover(my, target):
    tc = core[target]
    return len(my & tc) / len(tc) if tc else 0.0

def reachset(my, exclude):
    return {r for r in serv_roles if r != exclude and cover(my, r) >= COV_THR}

def greedy_bridge(role):
    me = set(core[role]); base = reachset(me, role)
    pool = set().union(*[core[r] for r in serv_roles if r != role]) - me - SKILL_STOP
    acquired, cur, opened = [], set(me), {}
    for _ in range(BRIDGE_STEPS):
        best, best_new = None, set()
        for sk in pool - set(acquired):
            new = reachset(cur | {sk}, role) - base - set(opened)
            if len(new) > len(best_new):
                best_new, best = new, sk
        if not best or not best_new:
            break
        acquired.append(best); cur |= {best}
        for r in best_new:
            opened[r] = list(acquired)
    return base, acquired, opened

print(f"Services core-skill nodes: {len(serv_roles)}  (TOPK={BRIDGE_TOPK}, coverage thr={COV_THR})\n")
connected = upskill = island = 0
for role in serv_roles:
    base, acq, opened = greedy_bridge(role)
    if base:
        connected += 1
        print(f"[{role}] CONNECTED — base reachable({len(base)}): {sorted(base)}")
        for r, need in opened.items():
            print(f"     + learn {need} -> opens '{r}'")
    elif opened:
        upskill += 1
        print(f"[{role}] REACHABLE-VIA-UPSKILL")
        for r, need in opened.items():
            print(f"     + learn {need} -> opens '{r}'")
    else:
        island += 1
        nn = max((r for r in serv_roles if r != role), key=lambda r: cover(set(core[role]), r))
        c = cover(set(core[role]), nn)
        if c >= ISLAND_FLOOR:
            print(f"[{role}] ISLAND — nearest '{nn}' ({c:.0%}); gap = {sorted(core[nn]-set(core[role]))[:5]}")
        else:
            print(f"[{role}] ISLAND — no meaningful adjacency (best {c:.0%}); specialist silo")

print(f"\nsummary: {connected} connected | {upskill} reachable-via-upskill | {island} island")
print("PASS if connected-cluster bridges are interpretable AND islands are labelled honestly.")


Services core-skill nodes: 20  (TOPK=12, coverage thr=0.3)

[SAP Consultant] REACHABLE-VIA-UPSKILL
     + learn ['sql'] -> opens 'Software Engineer (General)'
[Software Engineer (General)] CONNECTED — base reachable(3): ['Data / BI Analyst', 'Eng Manager / Tech Lead', 'Frontend Developer']
     + learn ['java'] -> opens 'Backend / Java Developer'
     + learn ['java'] -> opens 'Mobile Developer'
     + learn ['java'] -> opens 'Full Stack Developer'
     + learn ['java', 'sap abap development'] -> opens 'SAP Consultant'
     + learn ['java', 'sap abap development', 'dot'] -> opens '.NET Developer'
[Full Stack Developer] CONNECTED — base reachable(2): ['Backend / Java Developer', 'Frontend Developer']
     + learn ['web services'] -> opens 'Software Engineer (General)'
     + learn ['web services'] -> opens 'Eng Manager / Tech Lead'
     + learn ['web services'] -> opens 'Mobile Developer'
[Backend / Java Developer] CONNECTED — base reachable(3): ['Frontend Developer', 'Full Stack Develo

## 7 · Optional cross-check — unsupervised clustering (honours §8 "dual-field clustering")

Not required for the gates. KMeans on dual-field (title tokens + normalised skills) TF-IDF, to see whether comparable structure emerges *without* the hand-built canonical roles. If the clusters' dominant titles line up with the canonical roles above → stronger PASS. Skip if pressed for time.

In [27]:
from sklearn.cluster import KMeans

corpus = (sub["title"].fillna("").str.lower() + " " +
          sub["skills"].apply(lambda l: " ".join(x.replace(" ","_") for x in l)))
cvec = TfidfVectorizer(min_df=5, max_features=4000, sublinear_tf=True)
CX = cvec.fit_transform(corpus)

K = 18
km = KMeans(n_clusters=K, random_state=42, n_init=5).fit(CX)
sub["cluster"] = km.labels_
terms = np.array(cvec.get_feature_names_out())
print(f"KMeans k={K} — dominant titles + top terms per cluster:\n")
for c in range(K):
    mask = sub["cluster"]==c
    titles = sub.loc[mask,"title"].str.lower().str.replace(r'[^a-z ]',' ',regex=True)
    common = Counter(" ".join(titles).split()).most_common(4)
    centroid = km.cluster_centers_[c]
    topterms = [terms[i].replace('_',' ') for i in centroid.argsort()[::-1][:6]]
    print(f"  c{c:2d} (n={mask.sum():5d}) titles~{[w for w,_ in common]}  terms~{topterms}")


KMeans k=18 — dominant titles + top terms per cluster:

  c 0 (n= 1301) titles~['engineer', 'automation', 'test', 'qa']  terms~['selenium', 'automation testing', 'automation', 'software testing', 'test', 'engineer']
  c 1 (n=  976) titles~['application', 'developer', 'lead', 'engineer']  terms~['application development', 'development methodologies', 'application', 'css', 'developer', 'sap']
  c 2 (n=  431) titles~['data', 'engineer', 'big', 'senior']  terms~['scala', 'data', 'hadoop', 'spark', 'airflow', 'hive']
  c 3 (n= 1019) titles~['net', 'developer', 'dot', 'senior']  terms~['net', 'net core', 'dot', 'asp', 'developer', 'angular']
  c 4 (n= 1399) titles~['engineer', 'devops', 'senior', 'lead']  terms~['devops', 'kubernetes', 'continuous integration', 'ci cd', 'aws', 'docker']
  c 5 (n= 1747) titles~['developer', 'application', 'react', 'js']  terms~['javascript', 'html', 'css', 'react', 'developer', 'node']
  c 6 (n= 7509) titles~['developer', 'engineer', 'senior', 'lead']  terms~

## 8 · Verdict — paste these back

Copy the outputs of the gate cells back for review:
- **Gate 1** (cell under "GATE 1") — the probe-role nearest-neighbour block.
- **Gate 2** (cell under "GATE 2") — splittable-roles count + the distinctive-skill comparison (or the sparsity finding).
- **Gate 3** (cell under "GATE 3") — the per-role connected / upskill / island classification + the summary line.
- Plus the **threshold sensitivity table** (Step 5) and, if you ran it, the **KMeans cluster summary**.

Decision rule (from §8): all three gates PASS → proceed to stack choice + the legibility-first frontend. Clustering FAIL → the structure doesn't hold on this file; rethink before any app. Stratification sparse → record it as the honest finding and proceed (the within-stratum graph still stands).
